In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from pyspark.sql.functions import col
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [0]:
# Load the race results data
results_spark = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

display(results_spark)

In [0]:
# Keep only the variables used for modeling
model_spark = (
    results_spark
    .select(
        col("grid").cast("int").alias("grid"),
        col("laps").cast("int").alias("laps"),
        col("constructorId").cast("int").alias("constructorId"),
        col("positionOrder").cast("int").alias("positionOrder")
    )
    .dropna()
)

results_pd = model_spark.toPandas()

print("Shape of modeling data:", results_pd.shape)
results_pd.head()

In [0]:
# Keep only the variables used for modeling
model_spark = (
    results_spark
    .select(
        col("grid").cast("int").alias("grid"),
        col("laps").cast("int").alias("laps"),
        col("constructorId").cast("int").alias("constructorId"),
        col("positionOrder").cast("int").alias("positionOrder")
    )
    .dropna()
)

results_pd = model_spark.toPandas()

print("Shape of modeling data:", results_pd.shape)
results_pd.head()

In [0]:
# Define predictors and outcome
feature_cols = ["grid", "laps", "constructorId"]
target_col = "positionOrder"

X = results_pd[feature_cols]
y = results_pd[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [0]:
mlflow.end_run()

In [0]:
# 10 experiment settings
rf_settings = [
    {"n_estimators": 50,  "max_depth": 5,   "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5,   "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10,  "min_samples_split": 5,  "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 10,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 15,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 15,  "min_samples_split": 5,  "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 20,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": 20,  "min_samples_split": 10, "min_samples_leaf": 4},
    {"n_estimators": 500, "max_depth": None,"min_samples_split": 2,  "min_samples_leaf": 1}
]

experiment_rows = []

for run_idx, setting in enumerate(rf_settings, start=1):
    with mlflow.start_run(run_name=f"rf_trial_{run_idx}"):

        model = RandomForestRegressor(
            n_estimators=setting["n_estimators"],
            max_depth=setting["max_depth"],
            min_samples_split=setting["min_samples_split"],
            min_samples_leaf=setting["min_samples_leaf"],
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmse_val = mean_squared_error(y_test, preds) ** 0.5
        mae_val = mean_absolute_error(y_test, preds)
        r2_val = r2_score(y_test, preds)

        mlflow.log_params(setting)
        mlflow.log_metric("rmse", rmse_val)
        mlflow.log_metric("mae", mae_val)
        mlflow.log_metric("r2", r2_val)

        mlflow.sklearn.log_model(model, artifact_path="rf_model")

        experiment_rows.append({
            "run_number": run_idx,
            "n_estimators": setting["n_estimators"],
            "max_depth": setting["max_depth"],
            "min_samples_split": setting["min_samples_split"],
            "min_samples_leaf": setting["min_samples_leaf"],
            "rmse": rmse_val,
            "mae": mae_val,
            "r2": r2_val
        })

In [0]:
summary_df = pd.DataFrame(experiment_rows).sort_values(by="rmse").reset_index(drop=True)
display(summary_df)

In [0]:
# Refit the best model based on the experiment table
best_row = summary_df.iloc[0]

best_rf = RandomForestRegressor(
    n_estimators=int(best_row["n_estimators"]),
    max_depth=None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]),
    min_samples_split=int(best_row["min_samples_split"]),
    min_samples_leaf=int(best_row["min_samples_leaf"]),
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)
best_pred = best_rf.predict(X_test)

In [0]:
# Log artifacts for the best model in a separate run
with mlflow.start_run(run_name="best_rf_artifacts"):

    mlflow.log_param("best_n_estimators", int(best_row["n_estimators"]))
    mlflow.log_param("best_max_depth", None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]))
    mlflow.log_param("best_min_samples_split", int(best_row["min_samples_split"]))
    mlflow.log_param("best_min_samples_leaf", int(best_row["min_samples_leaf"]))

    mlflow.log_metric("best_rmse", float(best_row["rmse"]))
    mlflow.log_metric("best_mae", float(best_row["mae"]))
    mlflow.log_metric("best_r2", float(best_row["r2"]))

    mlflow.sklearn.log_model(best_rf, artifact_path="best_rf_model")

    # Feature importance plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(feature_cols, best_rf.feature_importances_)
    ax.set_xlabel("Importance Score")
    ax.set_title("Feature Importance for Best Random Forest")
    plt.tight_layout()
    fig.savefig("/tmp/feature_importance_rf.png")
    mlflow.log_artifact("/tmp/feature_importance_rf.png")
    plt.show()

    # Actual vs predicted plot
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    ax2.scatter(y_test, best_pred, alpha=0.35, s=12)
    lower_bound = min(y_test.min(), best_pred.min())
    upper_bound = max(y_test.max(), best_pred.max())
    ax2.plot([lower_bound, upper_bound], [lower_bound, upper_bound], linestyle="--", color="red")
    ax2.set_xlabel("Observed Position")
    ax2.set_ylabel("Predicted Position")
    ax2.set_title("Observed vs Predicted Values")
    plt.tight_layout()
    fig2.savefig("/tmp/observed_vs_predicted_rf.png")
    mlflow.log_artifact("/tmp/observed_vs_predicted_rf.png")
    plt.show()

    # Save predictions as csv
    pred_output = pd.DataFrame({
        "actual_position": y_test.values,
        "predicted_position": best_pred
    })
    pred_output.to_csv("/tmp/rf_predictions.csv", index=False)
    mlflow.log_artifact("/tmp/rf_predictions.csv")

In [0]:
from IPython.display import Image, display

display(Image("photo1.png"))

In [0]:
display(Image("photo2.png"))


Based on the experimental results, the best-performing model is the Random Forest with approximately 200 trees and a maximum depth of around 10.

This model achieves the lowest RMSE and MAE, along with one of the highest R² scores among all tested configurations. These metrics indicate that it provides the most accurate predictions while maintaining a good balance between bias and variance.
